# Using the Resume Feature in QMCPy

This notebook is the compact, recipe-style guide.
For background and a deeper discussion of the "how much accuracy do you need" question, see `demos/demo_resume_data/accuracy_and_resume.ipynb`.

The `resume` parameter lets you restart a stopped integration from where it left off.

**Typical workflow**:
1. Run `solution, data = sc.integrate()` with a loose tolerance for a quick estimate.
2. Save `data` to disk with `data.save(path)` so the state is preserved.
3. Later (or in a new session), load it: `data = Data.load(path)`.
4. Reconstruct a compatible solver (same problem definition and settings), tighten tolerance, and call `integrate(resume=data)`.

This demonstrates workflow and checkpointing correctness.
For small examples, wall-clock timing differences can be negligible; performance gains become clearer when the initial run already used substantial work.

In [1]:
from pathlib import Path

from qmcpy import CubQMCLatticeG, Genz, Lattice
from qmcpy.util.data import Data

In [2]:
TRACE_ITERATIONS = True  # Set to False to hide per-iteration logs during integrate().


def make_solver(abs_tol, rel_tol=0, seed=7, dimension=3, trace_label=""):
    integrand = Genz(
        Lattice(dimension=dimension, seed=seed),
        kind_func="oscillatory",
        kind_coeff=1,
    )
    solver = CubQMCLatticeG(integrand, abs_tol=abs_tol, rel_tol=rel_tol)
    solver.trace_iterations = TRACE_ITERATIONS
    solver.trace_label = trace_label
    return solver


def half_width(data):
    return (data.comb_bound_high.item() - data.comb_bound_low.item()) / 2


def print_stage_summary(rows):
    sep = "-" * 55
    print("Stage summary")
    print(sep)
    print(
        f"{'Stage':<7} {'abs_tol':>7} {'total n':>9} {'new n':>9} {'half-width':>10} {'time (s)':>8}"
    )
    print(sep)
    for name, abs_tol, total_n, new_n, hw, tsec in rows:
        print(
            f"{name:<7} {abs_tol:>7.0e} {int(total_n):>9,} {int(new_n):>9,} {hw:>10.2e} {tsec:>8.4f}"
        )
    print(sep)

## Step 1: Quick Estimate (Loose Tolerance)

Define a 3-D Genz oscillatory integral over the unit cube and run `CubQMCLatticeG` with a loose tolerance to get a fast initial estimate.

We fix the seed so notebook output is reproducible.

If you want iteration-by-iteration diagnostics (sample counts, `n_min`, `m`, shape growth, and interval half-width), set `TRACE_ITERATIONS = True` in the helper cell above.

In [3]:
abs_tol_loose = 1e-3
rel_tol = 0
solver = make_solver(abs_tol_loose, rel_tol=rel_tol, seed=7, dimension=3, trace_label="LOOSE")
solution1, data1 = solver.integrate()

print(f"Loose-tolerance solution:  {solution1.item():.8f}")
print(f"  Samples used:  {int(data1.n_total):,}")
print(f"  Time:          {data1.time_integrate:.4f} s")
print(f"  Estimated interval half-width: {half_width(data1):.2e}")

=== LOOSE iteration log ===
stage          solution    n_total      n_min    m      xfull.shape
------------------------------------------------------------------------
ITER         -0.4289211       1024          0   10        (1024, 3)
Loose-tolerance solution:  -0.42892111
  Samples used:  1,024
  Time:          0.0008 s
  Estimated interval half-width: 9.72e-04


## Step 2: Save the Integration State

`data1.save(path)` pickles the `Data` object to disk.
Pass `compress=True` to create a smaller gzip-compressed file (`.gz` suffix added automatically).

In [4]:
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)
save_path = output_dir / "resume_example_data1.pkl"
data1.save(save_path)
print(f"Saved to {save_path}  ({save_path.stat().st_size:,} bytes)")

# Compressed variant — the .gz suffix is added automatically
data1.save(output_dir / "resume_example_data1_compressed.pkl", compress=True)
save_path_compressed = output_dir / "resume_example_data1_compressed.pkl.gz"
gz_size = save_path_compressed.stat().st_size
print(f"Compressed file:  {gz_size:,} bytes  ({100*(1 - gz_size/save_path.stat().st_size):.0f}% smaller)")

Saved to output/resume_example_data1.pkl  (63,195 bytes)
Compressed file:  32,814 bytes  (48% smaller)


## Step 3: Resume with a Tighter Tolerance

Load the saved state and resume with a **compatible** solver (same integrand family, dimension, and randomization settings).
Only the incremental samples needed to meet the tighter tolerance are generated.

In [5]:
loaded_data = Data.load(save_path)
old_n_total = int(loaded_data.n_total)

abs_tol_tight = 1e-5
resume_solver = make_solver(abs_tol_tight, rel_tol=rel_tol, seed=7, dimension=3)
solution2, data2 = resume_solver.integrate(resume=loaded_data)

print(f"\nResumed solution:  {solution2.item():.8f}")
print(f"  Previous samples: {old_n_total:,}")
print(f"  Total samples:    {int(data2.n_total):,}")
print(f"  New samples:      {int(data2.n_total - old_n_total):,}")
print(f"  Time this step:   {data2.time_integrate:.4f} s")
print(f"  Estimated interval half-width: {half_width(data2):.2e}")

stage          solution    n_total      n_min    m      xfull.shape
------------------------------------------------------------------------
RESUME       -0.4289211       1024       1024   10        (1024, 3)
ITER         -0.4289211       1024       1024   10        (1024, 3)
ITER         -0.4289245       2048       1024   11        (2048, 3)
ITER         -0.4289312       4096       2048   12        (4096, 3)
ITER         -0.4289320       8192       4096   13        (8192, 3)
ITER         -0.4289320      16384       8192   14       (16384, 3)
ITER         -0.4289321      32768      16384   15       (32768, 3)

Resumed solution:  -0.42893212
  Previous samples: 1,024
  Total samples:    32,768
  New samples:      31,744
  Time this step:   0.0046 s
  Estimated interval half-width: 9.32e-06


## Step 4: Compare with a Fresh Run

For reference, solve from scratch with the same tight tolerance to see how much work was saved.

In [6]:
solver_fresh = make_solver(abs_tol_tight, rel_tol=rel_tol, seed=7, dimension=3, trace_label="FRESH")
solution3, data3 = solver_fresh.integrate()

print(f"\nFresh-start solution:  {solution3.item():.8f}")
print(f"  Samples used:        {int(data3.n_total):,}")
print(f"  Time:                {data3.time_integrate:.4f} s")
print()
print_stage_summary(
    [
        ("Loose", abs_tol_loose, data1.n_total, data1.n_total, half_width(data1), data1.time_integrate),
        (
            "Resumed",
            abs_tol_tight,
            data2.n_total,
            data2.n_total - old_n_total,
            half_width(data2),
            data2.time_integrate,
        ),
        ("Fresh", abs_tol_tight, data3.n_total, data3.n_total, half_width(data3), data3.time_integrate),
    ]
)
print(f"Solutions match: {abs(solution2.item() - solution3.item()) < 2 * abs_tol_tight}")

=== FRESH iteration log ===
stage          solution    n_total      n_min    m      xfull.shape
------------------------------------------------------------------------
ITER         -0.4289211       1024          0   10        (1024, 3)
ITER         -0.4289245       2048       1024   11        (2048, 3)
ITER         -0.4289312       4096       2048   12        (4096, 3)
ITER         -0.4289320       8192       4096   13        (8192, 3)
ITER         -0.4289320      16384       8192   14       (16384, 3)
ITER         -0.4289321      32768      16384   15       (32768, 3)

Fresh-start solution:  -0.42893212
  Samples used:        32,768
  Time:                0.0078 s

Stage summary
-------------------------------------------------------
Stage   abs_tol   total n     new n half-width time (s)
-------------------------------------------------------
Loose     1e-03     1,024     1,024   9.72e-04   0.0008
Resumed   1e-05    32,768    31,744   9.32e-06   0.0046
Fresh     1e-05    32,768    3

## Key Takeaways

The `resume` feature supports an iterative accuracy workflow:

1. Start with a loose tolerance to get a quick estimate.
2. Save the integration state.
3. Tighten the tolerance later.
4. Resume from the previous state instead of restarting.

This is most useful when the initial run already consumed meaningful work or when checkpointing across sessions is needed.
For small examples, wall-clock timing differences may be negligible; the main demonstrated benefit here is correctness of state reuse.